In [ ]:
from pathlib import Path

import pandas as pd

DATA = Path("../data")
OUTPUT = Path("../output")
OUTPUT.mkdir(exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

In [ ]:
accounts = pd.read_csv(DATA/"accounts.csv", parse_dates=["customer_since"])
subscriptions = pd.read_csv(DATA/"subscriptions.csv", parse_dates=["start_date", "end_date"])
monthly_usage = pd.read_csv(DATA/"monthly_usage.csv", parse_dates=["month"], date_format="%Y-%m")

frames = {"accounts": accounts, "subscriptions": subscriptions, "monthly_usage": monthly_usage}

for name, frame in frames.items():
    print(f"{name:<14}: {frame.shape}")

print("\nParsed date columns :")
for name, frame in frames.items():
    for column, dtype in frame.select_dtypes("datetime").dtypes.items():
        label = f"{name}.{column}"
        print(f"  {label:<25}: {dtype}")

print("\nColumn dtypes in subscriptions :")
print(subscriptions.dtypes.value_counts().to_string())

In [ ]:
for name, frame in frames.items():
    print(f"{name} : first 3 rows")
    display(frame.head(3))

In [ ]:
KEY_COLUMNS = {
    "accounts": ["account_id", "account_name", "segment", "region", "customer_since"],
    "subscriptions": ["subscription_id", "account_id", "product", "annual_contract_value", "end_date", "renewal_status"],
    "monthly_usage": ["subscription_id", "month", "seats_purchased", "active_users", "seat_utilization"],
}

for name, frame in frames.items():
    print(f"{name} : key columns")
    display(frame[KEY_COLUMNS[name]].head(3))

In [ ]:
acv = subscriptions["annual_contract_value"]

print(f"type   : {type(acv)}")
print(f"length : {len(acv)}")
print(f"sum    : {acv.sum():,.2f}")
print(f"max    : {acv.max():,.2f}")

In [ ]:
print("Renewal statuses :")
print(subscriptions["renewal_status"].value_counts().to_string())

print("\nSegment shares :")
print(accounts["segment"].value_counts(normalize=True).to_string())

In [ ]:
merged = subscriptions.merge(accounts, on="account_id", how="left", validate="many_to_one", indicator=True)

missing_account = (merged["_merge"] == "left_only").sum()
merged = merged.drop(columns="_merge")

print(f"subscriptions : {subscriptions.shape}")
print(f"accounts      : {accounts.shape}")
print(f"merged        : {merged.shape}")

print(f"\nRow count preserved : {len(merged) == len(subscriptions)}")
print(f"Subscriptions missing an account : {missing_account}")

In [ ]:
by_region = (
    merged.groupby("region")
    .agg(
        subscriptions=("subscription_id", "count"),
        total_acv=("annual_contract_value", "sum"),
        median_acv=("annual_contract_value", "median"),
        avg_utilization=("avg_seat_utilization", "mean"),
    )
    .sort_values("total_acv", ascending=False)
)

print("By region, sorted by total ACV :")
display(by_region.style.format({"total_acv": "${:,.2f}", "median_acv": "${:,.2f}", "avg_utilization": "{:.3f}"}))

In [ ]:
acv_pivot = merged.pivot_table(
    index="segment",
    columns="product",
    values="annual_contract_value",
    aggfunc="sum",
    fill_value=0,
)

print(f"Total ACV by segment and product {acv_pivot.shape} :")
display(acv_pivot.style.format("${:,.2f}"))